# smolagents + Ollama + mycontext DataAnalyzer

Run the **DataAnalyzer** pipeline **locally** with [Ollama](https://ollama.com/) — no API keys required.

**Prerequisites:**
1. Install [Ollama](https://ollama.com/) and start it: `ollama serve`
2. Pull a model: `ollama pull llama3.2` or `ollama pull qwen2:7b`
3. Install: `pip install "smolagents[toolkit]" mycontext-ai litellm`

**Flow:** Load CSV → build data description (pandas) → DataAnalyzer (Ollama) → visualizations

In [ ]:
import os
from pathlib import Path

# Register Ollama as a provider via LiteLLM.
# LiteLLM natively supports Ollama with the "ollama/" model prefix.
from mycontext.providers import register_provider
from mycontext.providers.litellm_provider import LiteLLMProvider

OLLAMA_MODEL = "qwen3-coder:latest"  # or "llama3.2", "qwen2:7b", "mistral", etc.

class OllamaProvider(LiteLLMProvider):
    def __init__(self, **kwargs):
        model = kwargs.pop("model", OLLAMA_MODEL)
        super().__init__(
            model=f"ollama/{model}" if not model.startswith("ollama/") else model,
            provider="ollama",
            api_key="ollama",
            **kwargs,
        )

register_provider("ollama", OllamaProvider)
print("Ollama provider registered. Run cells 1 and 2 before the agent.")

## 1. Load CSV and build data description (pandas)

Same as data_analyzer_csv_demo: load CSV with pandas and build description from `df.info()`, `df.describe()`, `df.head()`.

In [ ]:
import io
import pandas as pd
from pathlib import Path

CSV_PATH = Path("examples/sample_analysis_data.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("sample_analysis_data.csv")

df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape, "| Columns:", list(df.columns))

buf = io.StringIO()
df.info(buf=buf)
data_description = f"""Columns: {list(df.columns)}
Shape: {df.shape[0]} rows, {df.shape[1]} columns

Info:
{buf.getvalue()}

Describe:
{df.describe().to_string()}

Sample (first 5 rows):
{df.head().to_string()}"""

print("Data description length:", len(data_description), "chars")
df.head()

## 2. smolagents CodeAgent with Ollama

Use **OpenAIServerModel** with `api_base` pointing to Ollama. The agent calls `csv_to_data_description` and `run_data_analyzer`, then creates visualizations.

**Note:** RunDataAnalyzerTool uses `provider="ollama"` so DataAnalyzer runs on your local model.

In [ ]:
try:
    from smolagents import CodeAgent, OpenAIServerModel
    from smolagents.tools import Tool
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "smolagents[toolkit]"])
    from smolagents import CodeAgent, OpenAIServerModel
    from smolagents.tools import Tool

from pathlib import Path


class CsvToDataDescriptionTool(Tool):
    name = "csv_to_data_description"
    description = "Load a CSV file and produce a structured data description for DataAnalyzer. Args: csv_path (str). Returns data description."
    inputs = {"csv_path": {"type": "string", "description": "Path to the CSV file"}}
    output_type = "string"

    def forward(self, csv_path: str) -> str:
        import io
        import pandas as pd
        p = Path(csv_path)
        if not p.exists():
            return f"Error: file not found: {csv_path}"
        df_local = pd.read_csv(p)
        buf = io.StringIO()
        df_local.info(buf=buf)
        return f"""Columns: {list(df_local.columns)}\nShape: {df_local.shape[0]} rows, {df_local.shape[1]} columns\n\nInfo:\n{buf.getvalue()}\n\nDescribe:\n{df_local.describe().to_string()}\n\nSample:\n{df_local.head().to_string()}"""


class RunDataAnalyzerTool(Tool):
    name = "run_data_analyzer"
    description = "Run the DataAnalyzer template with a data description and goal. Args: data_description (str), goal (str), context (str, optional). Returns the analysis report."
    inputs = {
        "data_description": {"type": "string", "description": "Structured data description for DataAnalyzer"},
        "goal": {"type": "string", "description": "Analysis goal or question"},
        "context": {"type": "string", "description": "Optional extra context", "nullable": True},
    }
    output_type = "string"

    def forward(self, data_description: str, goal: str, context: str = "") -> str:
        from mycontext.templates.free.analysis import DataAnalyzer
        r = DataAnalyzer().execute(provider="ollama", data_description=data_description, goal=goal, context=context or None)
        return r.response


# Use Ollama: OpenAIServerModel with api_base pointing to local Ollama
model = OpenAIServerModel(
    model_id=OLLAMA_MODEL,
    api_base=OLLAMA_BASE,
    api_key="ollama",  # Ollama doesn't require auth; any string works
)

agent = CodeAgent(
    tools=[CsvToDataDescriptionTool(), RunDataAnalyzerTool()],
    model=model,
    additional_authorized_imports=["pathlib", "pandas", "matplotlib.pyplot"],
)

task = f"""1) Use csv_to_data_description(csv_path="{CSV_PATH}") to get the data description.
2) Use run_data_analyzer(data_description, goal, context) with goal='Identify trends, regional and product performance, anomalies, and give actionable recommendations.' and context='Monthly revenue and units by region and product. Cost column is COGS.'
3) Create Chart 1 (line chart for revenue trends by region), Chart 2 (bar chart for revenue/units by product/region), and a Dashboard (subplots for key metrics). Use df and plt; call plt.show() for each figure.
4) Call final_answer(report) with the analysis report text."""

print("Running agent with Ollama...")
result_agent = agent.run(task, additional_args={"df": df})
print("\n--- Agent final answer ---\n")
print(result_agent)